## Looking at the Preprocessed examples


### Patient Data 

In [ ]:
import torch
import numpy as np
import pandas as pd

import os
#os.chdir('C:/GU/Thesis/code')

In [8]:
data = pd.read_csv('cases_data.csv')

In [9]:
data.head()

,caseid,optype,sex,age,asa,bmi,preop_hb,preop_k,preop_na,preop_gluc,preop_cr,preop_alb,propofol,volatile,remifentanil
0,36,Breast,F,38.0,1.0,20.7,12.9,4.5,139.0,93.0,0.63,4.6,True,True,True
1,100,Colorectal,F,58.0,2.0,19.6,13.9,3.8,143.0,102.0,0.79,4.1,False,True,False
2,141,Breast,F,54.0,1.0,24.6,12.8,4.5,141.0,58.0,0.75,4.2,True,True,True
3,169,Thyroid,F,69.0,2.0,23.0,12.8,4.0,141.0,106.0,0.68,4.2,True,True,True
4,173,Colorectal,M,53.0,2.0,19.1,10.8,4.6,138.0,135.0,0.84,3.5,False,True,False


### Sequential Data

In [11]:
LOW_EDGES  = np.logspace(np.log10(0.5),  np.log10(4),  num=6)
MID_EDGES  = np.logspace(np.log10(4),    np.log10(13), num=6)
HIGH_EDGES = np.logspace(np.log10(13),   np.log10(50), num=6)

print(LOW_EDGES)
print(MID_EDGES)
print(HIGH_EDGES)

[0.5        0.75785828 1.14869835 1.74110113 2.63901582 4.        ]
[ 4.          5.06333502  6.40934037  8.11315939 10.269911   13.        ]
[13.         17.01957389 22.28199196 29.17153912 38.19132043 50.        ]


In [10]:
OUTPUT_DIR = 'preprocessed/eeg/'

# Load a single case
sample_path = os.path.join(OUTPUT_DIR, f'case_36.pt')
sample = torch.load(sample_path)

features = sample['features'].numpy()
bis      = sample['bis'].numpy()
caseid   = sample['caseid']

print(f"=== Case {caseid} ===")
print(f"  Features shape : {features.shape}  (timesteps x features)")
print(f"  BIS shape      : {bis.shape}")
print(f"  Duration       : {features.shape[0] / 60:.1f} minutes")

print(f"\n--- First 5 seconds ---")
print(f"  {'t':>4}  {'EEG_low(5bins)':>40}  {'EEG_mid(5bins)':>40}  {'EEG_high(5bins)':>40}  {'MAP':>6}  {'HR':>6}  {'SpO2':>6}  {'EtCO2':>6}  {'BIS':>6}")
for t in range(5):
    eeg_low  = np.array2string(features[t, 0:5],  precision=3, separator=',')
    eeg_mid  = np.array2string(features[t, 5:10], precision=3, separator=',')
    eeg_high = np.array2string(features[t, 10:15],precision=3, separator=',')
    map_val  = features[t, 15]
    hr_val   = features[t, 16]
    spo2_val = features[t, 17]
    etco2_val= features[t, 18]
    print(f"  {t:>4}  {eeg_low:>40}  {eeg_mid:>40}  {eeg_high:>40}  {map_val:>6.1f}  {hr_val:>6.1f}  {spo2_val:>6.1f}  {etco2_val:>6.1f}  {bis[t]:>6.1f}")

print(f"\n--- Sanity checks ---")
print(f"  NaN in features : {np.isnan(features).sum()} / {features.size}")
print(f"  NaN in BIS      : {np.isnan(bis).sum()} / {bis.size}")
print(f"  Features range  : [{np.nanmin(features):.3f}, {np.nanmax(features):.3f}]")
print(f"  BIS range       : [{np.nanmin(bis):.1f}, {np.nanmax(bis):.1f}]")
print(f"  EEG cols (0-14) : min={np.nanmin(features[:, 0:15]):.3f}, max={np.nanmax(features[:, 0:15]):.3f}")
print(f"  Vital cols(15-18): min={np.nanmin(features[:, 15:19]):.3f}, max={np.nanmax(features[:, 15:19]):.3f}")

=== Case 36 ===
  Features shape : (4477, 19)  (timesteps x features)
  BIS shape      : (4477,)
  Duration       : 74.6 minutes

--- First 5 seconds ---
     t                            EEG_low(5bins)                            EEG_mid(5bins)                           EEG_high(5bins)     MAP      HR    SpO2   EtCO2     BIS
     0           [7.24 ,7.566,7.75 ,6.545,7.603]           [5.758,5.367,4.182,3.634,2.802]           [2.062,2.249,2.692,2.519,2.283]    86.0    69.0   100.0     0.0    97.7
     1           [7.796,7.145,7.256,7.091,6.753]           [4.987,5.187,3.49 ,2.163,2.37 ]           [1.109,2.966,3.014,3.047,2.79 ]    86.0    69.0   100.0     0.0    97.7
     2           [9.245,8.898,7.467,7.688,6.198]           [4.494,4.687,4.328,3.824,2.5  ]           [1.933,2.4  ,2.269,3.168,2.625]    86.0    69.0   100.0     0.0    97.7
     3      [10.48 ,10.327, 8.293, 8.597, 6.746]           [4.361,2.159,3.101,2.534,2.526]           [2.742,2.421,2.125,2.403,1.321]    86.0    69.0   100